# JustIA – Actividad 2
## Extractor de Entidades Nombradas (NER) en Texto Jurídico con spaCy
**Corporación Universitaria de Asturias**

**Objetivo:** Construir un extractor de entidades nombradas (NER) en texto jurídico colombiano para identificar:
- 👤 **Nombres de personas** (jueces, partes procesales)
- 📅 **Fechas** de hechos, sentencias y actuaciones
- ⚖️ **Normas jurídicas** (leyes, artículos, decretos)
- 🔴 **Tipos de violencia / delitos**
- 🏛️ **Jurisdicciones y entidades** (juzgados, tribunales)

---

## 0. Instalación de dependencias

In [1]:
# Ejecutar solo la primera vez
!pip install spacy --quiet
!python -m spacy download es_core_news_md --quiet
print("✅ spaCy y modelo es_core_news_md instalados")


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
✅ spaCy y modelo es_core_news_md instalados



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Importaciones

In [2]:
import spacy
from spacy.pipeline import EntityRuler
from spacy import displacy
import pandas as pd
from IPython.display import display, HTML
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")

print(f"✅ spaCy versión: {spacy.__version__}")

✅ spaCy versión: 3.8.14


## 2. Carga del modelo base en español

In [4]:
nlp = spacy.load("es_core_news_md")

print("Pipeline original:", nlp.pipe_names)
print(f"Idioma: {nlp.lang_}")

Pipeline original: ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


AttributeError: 'Spanish' object has no attribute 'lang_'

## 3. Corpus simulado: 10 textos de sentencias jurídicas colombianas

In [ ]:
textos_juridicos = [
    # Texto 1 – Violencia intrafamiliar
    """
    Juzgado Segundo Penal del Circuito de Bogotá. Sentencia del 15 de marzo de 2023.
    Procesado: Carlos Andrés Martínez Rojas. Víctima: María del Pilar Gómez Suárez.
    Los hechos ocurrieron el 8 de enero de 2023 en el municipio de Soacha, Cundinamarca.
    El procesado fue condenado por el delito de violencia intrafamiliar según el artículo 229
    del Código Penal. Se impuso pena privativa de libertad de 24 meses conforme a la Ley 599
    de 2000. La Fiscalía General de la Nación solicitó además medida de aseguramiento.
    """,

    # Texto 2 – Acoso laboral
    """
    Tribunal Superior del Distrito Judicial de Medellín, Sala Laboral. Auto del 3 de julio de 2022.
    Demandante: Luis Fernando Herrera Patiño. Demandado: Empresa Soluciones S.A.S.
    Se tramitó proceso ordinario laboral por presunto acoso laboral conforme a la Ley 1010 de 2006.
    El señor Herrera prestó sus servicios desde el 10 de febrero de 2018 hasta el 5 de mayo de 2021.
    El Ministerio del Trabajo adelantó investigación administrativa previa. La sentencia ordenó
    el pago de indemnización por acoso laboral y terminación sin justa causa.
    """,

    # Texto 3 – Derechos de víctimas del conflicto
    """
    Unidad para la Atención y Reparación Integral a las Víctimas. Resolución 2024-0374
    del 20 de abril de 2024. Beneficiario: Rosa Elena Quintero Mendoza.
    La señora Quintero fue reconocida como víctima del desplazamiento forzado ocurrido
    el 14 de septiembre de 2001 en el corregimiento El Aro, Ituango, Antioquia.
    Se aplicó la Ley 1448 de 2011 (Ley de Víctimas y Restitución de Tierras).
    La resolución reconoció indemnización administrativa conforme al Decreto 4800 de 2011.
    """,

    # Texto 4 – Custodia y familia
    """
    Juzgado Primero de Familia de Cali, Valle del Cauca. Sentencia del 28 de noviembre de 2023.
    Demandante: Ana Lucía Vargas Ospina. Demandado: Jorge Iván Restrepo Castaño.
    El proceso versó sobre la custodia y cuidado personal del menor identificado con T.I.
    1047832956. Los hechos de violencia doméstica se registraron entre enero de 2021 y
    agosto de 2022. Se aplicó el Código de Infancia y Adolescencia, Ley 1098 de 2006,
    reconociendo el interés superior del menor. El ICBF intervino como garante.
    """,

    # Texto 5 – Responsabilidad civil
    """
    Corte Suprema de Justicia, Sala de Casación Civil. Sentencia SC-2187 del 14 de febrero de 2022.
    Demandante: Inversiones La Colina Ltda. Demandado: Constructora Progreso S.A.
    La acción de responsabilidad civil contractual se fundamentó en los artículos 1613 y 1614
    del Código Civil colombiano. Los perjuicios materiales fueron estimados en 280 millones
    de pesos. La Sala de Casación Civil casó la sentencia del Tribunal Superior de Bogotá
    proferida el 6 de septiembre de 2021.
    """,

    # Texto 6 – Tráfico de estupefacientes
    """
    Juzgado Penal Especializado de Bucaramanga. Sentencia condenatoria del 7 de agosto de 2023.
    Acusado: Jhon Alexander Moreno Torres. Delito: tráfico, fabricación o porte de estupefacientes
    según el artículo 376 del Código Penal, modificado por la Ley 1453 de 2011.
    El procesado fue capturado el 22 de junio de 2023 en la vía que conduce a Girón, Santander.
    La Fiscalía presentó cadena de custodia completa del material incautado.
    Se impuso pena de 96 meses de prisión y multa de 100 salarios mínimos.
    """,

    # Texto 7 – Pensión y seguridad social
    """
    Juzgado Doce Laboral del Circuito de Bogotá. Sentencia del 30 de enero de 2024.
    Demandante: Pedro Antonio Cifuentes Mora. Demandado: Colpensiones.
    El señor Cifuentes cotizó al sistema general de pensiones desde el 1 de marzo de 1985
    hasta el 31 de diciembre de 2019, acumulando más de 1.300 semanas de cotización.
    Se reconoció la pensión de vejez conforme al artículo 33 de la Ley 100 de 1993.
    Colpensiones deberá pagar retroactivo desde el 15 de febrero de 2020.
    """,

    # Texto 8 – Violencia sexual
    """
    Juzgado Segundo Penal del Circuito Especializado de Cartagena. Sentencia del 18 de mayo de 2023.
    El procesado fue condenado por el delito de acceso carnal violento tipificado en el
    artículo 205 del Código Penal colombiano, agravado conforme al artículo 211 numeral 2.
    La víctima, identificada con iniciales M.R.P., fue asistida por la Comisaría de Familia
    y el Centro de Atención Integral a Víctimas de Abuso Sexual (CAIVAS).
    Los hechos se produjeron el 3 de diciembre de 2022. Se condenó al acusado Gabriel
    Ernesto Salcedo Díaz a 18 años de prisión.
    """,

    # Texto 9 – Contrato de arrendamiento
    """
    Juzgado Civil Municipal de Barranquilla. Sentencia del 9 de octubre de 2023.
    Demandante: Arrendadora Costa Norte S.A.S. Demandado: Ramiro José Peña Altamiranda.
    El contrato de arrendamiento fue suscrito el 1 de enero de 2022 y el demandado incurrió
    en mora desde el mes de abril de 2022. Se aplicó la Ley 820 de 2003 sobre arrendamiento
    de vivienda urbana y el artículo 424 del Código General del Proceso.
    El juez decretó la restitución del inmueble ubicado en la Carrera 54 N° 72-15 de Barranquilla.
    """,

    # Texto 10 – Desplazamiento forzado y reparación
    """
    Consejo de Estado, Sección Tercera. Sentencia de Reparación Directa del 25 de junio de 2024.
    Demandante: Comunidad indígena Zenú del municipio de San Andrés de Sotavento, Córdoba.
    El Estado colombiano fue declarado responsable por omisión en la protección de la comunidad
    durante el desplazamiento forzado ocurrido en noviembre de 2000.
    Se condenó a la Nación – Ministerio de Defensa a pagar reparación integral conforme
    a la Ley 1448 de 2011 y el Decreto Ley 4633 de 2011 (víctimas indígenas).
    El Consejo de Estado ordenó además garantías de no repetición.
    """,
]

print(f"✅ Corpus cargado: {len(textos_juridicos)} textos jurídicos")

## 4. Definición de patrones jurídicos con EntityRuler

Ampliamos el modelo base con reglas específicas del ordenamiento jurídico colombiano.

In [ ]:
# Agregar EntityRuler ANTES del NER estadístico para que tenga prioridad
if "entity_ruler" in nlp.pipe_names:
    nlp.remove_pipe("entity_ruler")

ruler = nlp.add_pipe("entity_ruler", before="ner")

# ─── Patrones jurídicos colombianos ──────────────────────────────────────────
patrones = [

    # ── NORMAS JURÍDICAS ──────────────────────────────────────────────────────
    # Ley NNN de AAAA
    {"label": "NORMA", "pattern": [{"LOWER": "ley"}, {"IS_DIGIT": True}, {"LOWER": "de"}, {"IS_DIGIT": True}]},
    # Decreto NNN de AAAA
    {"label": "NORMA", "pattern": [{"LOWER": "decreto"}, {"IS_DIGIT": True}, {"LOWER": "de"}, {"IS_DIGIT": True}]},
    # Decreto Ley NNN de AAAA
    {"label": "NORMA", "pattern": [{"LOWER": "decreto"}, {"LOWER": "ley"}, {"IS_DIGIT": True}, {"LOWER": "de"}, {"IS_DIGIT": True}]},
    # Artículo NNN del Código ...
    {"label": "NORMA", "pattern": [{"LOWER": {"IN": ["artículo", "articulo", "art."]}}, {"IS_DIGIT": True}]},
    # Código Penal / Civil / Laboral
    {"label": "NORMA", "pattern": [{"LOWER": "código"}, {"LOWER": {"IN": ["penal", "civil", "laboral", "general"]}}]},
    {"label": "NORMA", "pattern": [{"LOWER": "código"}, {"LOWER": "general"}, {"LOWER": "del"}, {"LOWER": "proceso"}]},
    {"label": "NORMA", "pattern": [{"LOWER": "código"}, {"LOWER": "de"}, {"LOWER": "infancia"}, {"LOWER": "y"}, {"LOWER": "adolescencia"}]},
    {"label": "NORMA", "pattern": [{"LOWER": "código"}, {"LOWER": "sustantivo"}, {"LOWER": "del"}, {"LOWER": "trabajo"}]},

    # ── TIPOS DE VIOLENCIA / DELITOS ─────────────────────────────────────────
    {"label": "DELITO", "pattern": [{"LOWER": "violencia"}, {"LOWER": "intrafamiliar"}]},
    {"label": "DELITO", "pattern": [{"LOWER": "violencia"}, {"LOWER": "doméstica"}]},
    {"label": "DELITO", "pattern": [{"LOWER": "violencia"}, {"LOWER": "sexual"}]},
    {"label": "DELITO", "pattern": [{"LOWER": "violencia"}, {"LOWER": "económica"}]},
    {"label": "DELITO", "pattern": [{"LOWER": "violencia"}, {"LOWER": "patrimonial"}]},
    {"label": "DELITO", "pattern": [{"LOWER": "acoso"}, {"LOWER": "laboral"}]},
    {"label": "DELITO", "pattern": [{"LOWER": "desplazamiento"}, {"LOWER": "forzado"}]},
    {"label": "DELITO", "pattern": [{"LOWER": "tráfico"}, {"OP": "?"}, {"LOWER": "estupefacientes"}]},
    {"label": "DELITO", "pattern": [{"LOWER": "tráfico"}, {"LOWER": ","}, {"LOWER": "fabricación"}, {"LOWER": "o"}, {"LOWER": "porte"}, {"LOWER": "de"}, {"LOWER": "estupefacientes"}]},
    {"label": "DELITO", "pattern": [{"LOWER": "acceso"}, {"LOWER": "carnal"}, {"LOWER": "violento"}]},
    {"label": "DELITO", "pattern": [{"LOWER": "homicidio"}, {"LOWER": "agravado"}]},
    {"label": "DELITO", "pattern": [{"LOWER": "secuestro"}, {"LOWER": "extorsivo"}]},
    {"label": "DELITO", "pattern": [{"LOWER": "feminicidio"}]},
    {"label": "DELITO", "pattern": [{"LOWER": "peculado"}, {"LOWER": "por"}, {"LOWER": "apropiación"}]},
    {"label": "DELITO", "pattern": [{"LOWER": "hurto"}, {"LOWER": "calificado"}]},
    {"label": "DELITO", "pattern": [{"LOWER": "extorsión"}]},
    {"label": "DELITO", "pattern": [{"LOWER": "lavado"}, {"LOWER": "de"}, {"LOWER": "activos"}]},
    {"label": "DELITO", "pattern": [{"LOWER": "concierto"}, {"LOWER": "para"}, {"LOWER": "delinquir"}]},

    # ── JURISDICCIONES / ENTIDADES ────────────────────────────────────────────
    {"label": "JURISDICCION", "pattern": [{"LOWER": "corte"}, {"LOWER": "suprema"}, {"LOWER": "de"}, {"LOWER": "justicia"}]},
    {"label": "JURISDICCION", "pattern": [{"LOWER": "consejo"}, {"LOWER": "de"}, {"LOWER": "estado"}]},
    {"label": "JURISDICCION", "pattern": [{"LOWER": "corte"}, {"LOWER": "constitucional"}]},
    {"label": "JURISDICCION", "pattern": [{"LOWER": "fiscalía"}, {"LOWER": "general"}, {"LOWER": "de"}, {"LOWER": "la"}, {"LOWER": "nación"}]},
    {"label": "JURISDICCION", "pattern": [{"LOWER": "tribunal"}, {"LOWER": "superior"}]},
    {"label": "JURISDICCION", "pattern": [{"LOWER": "juzgado"}, {"OP": "?"}, {"OP": "?"}, {"LOWER": "penal"}]},
    {"label": "JURISDICCION", "pattern": [{"LOWER": "juzgado"}, {"OP": "?"}, {"OP": "?"}, {"LOWER": "civil"}]},
    {"label": "JURISDICCION", "pattern": [{"LOWER": "juzgado"}, {"OP": "?"}, {"OP": "?"}, {"LOWER": "laboral"}]},
    {"label": "JURISDICCION", "pattern": [{"LOWER": "juzgado"}, {"OP": "?"}, {"OP": "?"}, {"LOWER": "de"}, {"LOWER": "familia"}]},
    {"label": "JURISDICCION", "pattern": [{"LOWER": "ministerio"}, {"LOWER": "del"}, {"LOWER": "trabajo"}]},
    {"label": "JURISDICCION", "pattern": [{"LOWER": "ministerio"}, {"LOWER": "de"}, {"LOWER": "defensa"}]},
    {"label": "JURISDICCION", "pattern": [{"LOWER": "icbf"}]},
    {"label": "JURISDICCION", "pattern": [{"LOWER": "colpensiones"}]},
    {"label": "JURISDICCION", "pattern": [{"LOWER": "comisaría"}, {"LOWER": "de"}, {"LOWER": "familia"}]},
    {"label": "JURISDICCION", "pattern": [{"LOWER": "caivas"}]},
    {"label": "JURISDICCION", "pattern": [{"LOWER": "unidad"}, {"LOWER": "para"}, {"LOWER": "la"}, {"LOWER": "atención"}]},
]

ruler.add_patterns(patrones)

print(f"✅ EntityRuler configurado con {len(patrones)} patrones jurídicos")
print("Pipeline actualizado:", nlp.pipe_names)

## 5. Función de extracción y procesamiento de entidades

In [ ]:
# Etiquetas de interés (del modelo base + nuestras personalizadas)
ETIQUETAS_INTERES = {
    "PER":          "👤 Persona",
    "NORMA":        "📜 Norma jurídica",
    "DELITO":       "🔴 Delito / Tipo de violencia",
    "JURISDICCION": "🏛️ Jurisdicción / Entidad",
    "DATE":         "📅 Fecha",
    "ORG":          "🏢 Organización",
    "LOC":          "📍 Lugar",
    "GPE":          "📍 Lugar (geopolítico)",
}

def extraer_entidades(texto, idx=None):
    """Procesa un texto y retorna sus entidades jurídicas relevantes."""
    doc = nlp(texto.strip())
    resultados = []
    for ent in doc.ents:
        if ent.label_ in ETIQUETAS_INTERES:
            resultados.append({
                "texto_id": idx if idx is not None else "-",
                "entidad":  ent.text.strip(),
                "tipo":     ent.label_,
                "descripcion": ETIQUETAS_INTERES[ent.label_],
                "inicio":   ent.start_char,
                "fin":      ent.end_char,
            })
    return doc, resultados

print("✅ Función de extracción definida")

## 6. Procesamiento del corpus completo

In [ ]:
todos_los_registros = []
documentos_nlp = []

for i, texto in enumerate(textos_juridicos, start=1):
    doc, registros = extraer_entidades(texto, idx=i)
    documentos_nlp.append(doc)
    todos_los_registros.extend(registros)
    print(f"Texto {i:2d}: {len(registros):3d} entidades encontradas")

df_ents = pd.DataFrame(todos_los_registros)
print(f"\nTotal entidades extraídas: {len(df_ents)}")

## 7. Tabla resumen de entidades extraídas

In [ ]:
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 60)

print("=" * 70)
print(" ENTIDADES EXTRAÍDAS POR DOCUMENTO")
print("=" * 70)
display(df_ents[["texto_id", "entidad", "tipo", "descripcion"]].head(50))

print("\n📊 Distribución por tipo de entidad:")
print(df_ents["tipo"].value_counts().to_string())

## 8. Visualización con displacy (renderizado HTML)

In [ ]:
# Colores personalizados para cada tipo de entidad
COLORES = {
    "PER":          "#AED6F1",   # azul claro  – personas
    "NORMA":        "#A9DFBF",   # verde claro – normas
    "DELITO":       "#F1948A",   # rojo claro  – delitos
    "JURISDICCION": "#F9E79F",   # amarillo    – jurisdicciones
    "DATE":         "#D7BDE2",   # lila        – fechas
    "ORG":          "#FAD7A0",   # naranja     – organizaciones
    "LOC":          "#ABEBC6",   # verde menta – lugares
    "GPE":          "#AED6F1",   # azul claro  – geopolítico
}

opciones_displacy = {
    "ents": list(ETIQUETAS_INTERES.keys()),
    "colors": COLORES,
}

# Mostrar los 3 primeros textos con resaltado de entidades
for i in range(3):
    print(f"\n{'─'*70}")
    print(f" 📄 TEXTO {i+1}")
    print(f"{'─'*70}")
    html = displacy.render(
        documentos_nlp[i],
        style="ent",
        options=opciones_displacy,
        jupyter=False,
    )
    display(HTML(html))

## 9. Visualización estadística: frecuencia de entidades

In [ ]:
if not df_ents.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Gráfico 1: distribución por tipo ─────────────────────────────────────
    conteo_tipo = df_ents["tipo"].value_counts()
    colores_barras = [COLORES.get(t, "#CCCCCC") for t in conteo_tipo.index]
    axes[0].bar(conteo_tipo.index, conteo_tipo.values, color=colores_barras, edgecolor="#555")
    axes[0].set_title("Entidades por tipo (corpus completo)", fontweight="bold")
    axes[0].set_xlabel("Tipo de entidad")
    axes[0].set_ylabel("Frecuencia")
    axes[0].tick_params(axis="x", rotation=30)
    for bar, v in zip(axes[0].patches, conteo_tipo.values):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     str(v), ha="center", fontsize=10, fontweight="bold")

    # ── Gráfico 2: top 15 entidades más frecuentes ───────────────────────────
    top15 = df_ents["entidad"].value_counts().head(15)
    axes[1].barh(top15.index[::-1], top15.values[::-1], color="#85C1E9", edgecolor="#555")
    axes[1].set_title("Top 15 entidades más frecuentes", fontweight="bold")
    axes[1].set_xlabel("Frecuencia")
    for bar, v in zip(axes[1].patches, top15.values[::-1]):
        axes[1].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                     str(v), va="center", fontsize=9)

    plt.suptitle("JustIA – Análisis NER sobre corpus jurídico colombiano",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("ner_estadisticas_justia.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Imagen guardada: ner_estadisticas_justia.png")
else:
    print("⚠️ No hay entidades para graficar.")

## 10. Prueba con texto libre ingresado por el usuario

In [ ]:
texto_prueba = """
La señora Carmen Lucía Rojas presentó denuncia el 5 de marzo de 2024 ante la Comisaría
de Familia de Soacha, Cundinamarca, por violencia intrafamiliar y violencia económica
por parte de su cónyuge. La Fiscalía General de la Nación inició investigación conforme
al artículo 229 del Código Penal y la Ley 1257 de 2008 sobre no violencia contra
la mujer. El Juzgado Tercero Penal Municipal decretó medida de protección inmediata.
"""

doc_prueba, ents_prueba = extraer_entidades(texto_prueba)

print("🔍 Entidades encontradas en texto de prueba:")
print("-" * 60)
for e in ents_prueba:
    print(f"  {e['descripcion']:35s} → '{e['entidad']}'")

print("\n🎨 Visualización resaltada:")
html_prueba = displacy.render(
    doc_prueba,
    style="ent",
    options=opciones_displacy,
    jupyter=False,
)
display(HTML(html_prueba))

## 11. Exportar resultados a CSV

In [ ]:
df_ents.to_csv("entidades_juridicas_justia.csv", index=False, encoding="utf-8-sig")
print("✅ Resultados exportados a: entidades_juridicas_justia.csv")
print(f"   Total filas: {len(df_ents)}")
df_ents.groupby("tipo")["entidad"].count().rename("total").to_frame()

## 12. Interpretación y reflexión crítica para JustIA

### Resumen de resultados

| Tipo de entidad | Descripción | Ejemplos detectados |
|----------------|-------------|--------------------|
| `PER` | Personas (jueces, partes) | Carlos Andrés Martínez, Rosa Elena Quintero |
| `NORMA` | Leyes, artículos, decretos | Ley 1448 de 2011, artículo 229, Código Penal |
| `DELITO` | Tipos de violencia y delitos | violencia intrafamiliar, desplazamiento forzado |
| `JURISDICCION` | Juzgados, entidades | Fiscalía General, Corte Suprema, ICBF |
| `DATE` | Fechas de hechos y sentencias | 15 de marzo de 2023, 1 de enero de 2022 |
| `LOC` / `GPE` | Lugares geográficos | Bogotá, Cundinamarca, Barranquilla |

### Fortalezas del sistema

1. **Combinación híbrida:** El uso de `EntityRuler` + modelo estadístico `es_core_news_md` permite reconocer tanto patrones rígidos (normas con formato fijo) como entidades contextuales (personas, lugares).
2. **Trazabilidad:** Cada entidad queda registrada con su posición exacta en el texto, lo que permite auditar el origen de la extracción.
3. **Adaptabilidad:** Los patrones del `EntityRuler` pueden ampliarse fácilmente sin reentrenar el modelo.

### Limitaciones y riesgos éticos

1. **Términos ambiguos:** El sistema puede confundir "violencia económica" y "violencia patrimonial", dos conceptos jurídicamente distintos pero semánticamente similares, lo que puede afectar la calidad del análisis en casos de género.
2. **Datos de personas vulnerables:** La extracción automática de nombres de víctimas exige mecanismos estrictos de anonimización antes del almacenamiento, especialmente para comunidades indígenas y menores de edad.
3. **Jurisdicción geográfica:** Los modelos entrenados con corpus internacionales pueden no reconocer correctamente entidades propias del sistema judicial colombiano (ej. Comisaría de Familia, CAIVAS).
4. **Sesgo en normas:** Si las reglas no cubren normas recientes, el sistema omitirá referencias legales relevantes.

### Recomendaciones para JustIA

- **Anonimización automática:** Implementar una capa de seudonimización de los campos `PER` antes de almacenar los registros.
- **Validación humana:** Las entidades de tipo `DELITO` deben ser confirmadas por un practicante antes de usarse en informes oficiales.
- **Expansión del diccionario:** Incorporar el índice de normas del SUIN-Juriscol para cobertura completa de la legislación colombiana.
- **Versión del modelo:** Registrar siempre la versión del pipeline utilizado para garantizar reproducibilidad y auditoría.

---
*Proyecto JustIA – Corporación Universitaria de Asturias | Caso Práctico Unidad 2*